# Морфологическая разметка

In [3]:
import pandas as pd

In [ ]:
!pip install stanza

In [2]:
import stanza

In [ ]:
stanza.download('ru')
nlp = stanza.Pipeline('ru', processors='tokenize,pos,lemma')

In [60]:
import numpy as np

In [61]:
from tqdm import tqdm

In [62]:
import ast as ast

In [63]:
df = pd.read_csv('combined_results.csv')

In [64]:
df.head()

,sentence_id,sentence_text,word,probability,token_probabilities,token_count,depth,tokens,token_texts,stop_reason,separator_probability,source_file
0,1,В тот момент <mask>,ы,0.08386,[np.float16(0.08386)],1,1,[4655],['ы'],{' '},0.020264,pred_20251007_181120_s1_В тот момент <mask>.csv
1,1,В тот момент <mask>,в,0.02347,[np.float16(0.02347)],1,1,[5591],['в'],{' '},0.193970,pred_20251007_181120_s1_В тот момент <mask>.csv
2,1,В тот момент <mask>,в,0.01600,[np.float16(0.016)],1,1,[111072],['\xa0в'],{' '},0.023636,pred_20251007_181120_s1_В тот момент <mask>.csv
3,1,В тот момент <mask>,а,0.01347,[np.float16(0.01347)],1,1,[1506],['а'],{' '},0.110474,pred_20251007_181120_s1_В тот момент <mask>.csv
4,1,В тот момент <mask>,В,0.01327,[np.float16(0.01327)],1,1,[16604],['В'],{' '},0.113342,pred_20251007_181120_s1_В тот момент <mask>.csv


In [65]:
df['sentence_number'] = df.groupby('sentence_id').ngroup() + 1

In [66]:
def get_stanza_analysis(word):
    """ функция, которая производит морф анализ предсказаний"""
    doc = nlp(word)
    word_info = doc.sentences[0].words[0]
    return {
        'upos_word': word_info.upos,
        'lemma_word': word_info.lemma,
        'feats': word_info.feats
    }

In [ ]:
tqdm.pandas(desc="Обработка слов")

stanza_results = df['word'].progress_apply(get_stanza_analysis)

In [69]:
df[['upos_word', 'lemma_word', 'feats']] = pd.DataFrame(stanza_results.tolist())

In [70]:
df_people = pd.read_csv('people_with_prob.csv')

In [71]:
df_people.head()

,Unnamed: 0,word.id,Lemma,POS,mapped_POS,Form,Left context,answer,lemma_answer,upos_answer,...,feats_answer_split,features_accuracy,cloze_accuracy,lemma_accuracy,pos_accuracy,item.id,word.serial.no,subject.id,is_russian,probability_y
0,0,атмосфера,атмосфера,S,NOUN,"S,жен,неод=им,ед",В тот момент,когда,когда,ADV,...,{'Degree=Pos'},[],0,0,0,59,4,DKC573,True,0.1250
1,1,атмосфера,атмосфера,S,NOUN,"S,жен,неод=им,ед",В тот момент,он,он,PRON,...,"{'PronType=Prs', 'Case=Nom', 'Gender=Masc', 'P...","['Case=Nom', 'Number=Sing']",0,0,0,59,4,MQG816,True,0.2500
2,2,атмосфера,атмосфера,S,NOUN,"S,жен,неод=им,ед",В тот момент,я,я,PRON,...,"{'Person=1', 'Case=Nom', 'Number=Sing', 'PronT...","['Case=Nom', 'Number=Sing']",0,0,0,59,4,VTN703,True,0.0625
3,3,атмосфера,атмосфера,S,NOUN,"S,жен,неод=им,ед",В тот момент,он,он,PRON,...,"{'PronType=Prs', 'Case=Nom', 'Gender=Masc', 'P...","['Case=Nom', 'Number=Sing']",0,0,0,59,4,DLV943,True,0.2500
4,4,атмосфера,атмосфера,S,NOUN,"S,жен,неод=им,ед",В тот момент,появился,появиться,VERB,...,"{'Tense=Past', 'Mood=Ind', 'Voice=Mid', 'VerbF...",['Number=Sing'],0,0,0,59,4,FYK814,True,0.0625


In [72]:
def convert_feats(feats):
    if pd.isna(feats):
        return set()

    if isinstance(feats, set):
        return feats

    if isinstance(feats, str):
        result = ast.literal_eval(feats)
        if isinstance(result, set):
            return result
        elif isinstance(result, list):
            return set(result)
        elif isinstance(result, dict):
            return set(result.keys()) if result else set()

    return set(feats)

df_people['mapped_feats_ud_processed'] = df_people['mapped_feats_ud'].apply(convert_feats)

In [73]:
df['stanza_results'] = stanza_results

In [74]:
target_dict = {}
for _, row in df_people.iterrows():
    left_context = row['Left context']
    target_dict[left_context] = {
        'word_id': row['word.id'],
        'Lemma': row['Lemma'],
        'mapped_POS': row['mapped_POS'],
        'mapped_feats_ud': row['mapped_feats_ud']
    }

In [75]:
def find_target_word(sentence_text):
    """подготовительная фукция, чтобы потом можно было из предложений с <mask> взять нужный контекст"""
    if '<mask>' in sentence_text:
        left_context = sentence_text.split('<mask>')[0].strip()
        return target_dict.get(left_context, {})
    return {}

In [76]:
# Добавление таргетных слов

target_info = df['sentence_text'].apply(find_target_word)
df[['target_word_id', 'target_Lemma', 'target_mapped_POS', 'target_mapped_feats_ud']] = pd.DataFrame(target_info.tolist())

In [77]:
def get_feature_accuracy(row):
    """для пересечения морф. признаков таргета и предсказаний"""
    target_feats = row['target_mapped_feats_ud']
    stanza_feats_str = row['stanza_results']['feats']

    if stanza_feats_str is None:
        return []

    stanza_feats = set()
    if '|' in stanza_feats_str:
        for feat in stanza_feats_str.split('|'):
            stanza_feats.add(feat)
    else:
        stanza_feats.add(stanza_feats_str)

    if isinstance(target_feats, str):
        if target_feats.startswith('{'):
            target_feats_set = eval(target_feats)
        else:
            target_feats_set = set(target_feats.split('|') if '|' in target_feats else [target_feats])
    else:
        target_feats_set = target_feats

    if not isinstance(target_feats_set, set):
        target_feats_set = set(target_feats_set)

    intersection = list(target_feats_set.intersection(stanza_feats))
    return intersection


In [78]:
# Сначала получаем пересечения
df['feature_accuracy'] = df.apply(get_feature_accuracy, axis=1)

# полное совпадение предсказания и таргета
df['cloze_accuracy'] = (df['word'].str.lower() == df['target_word_id'].str.lower()).astype(int)

# совпадение лемм предсказания и таргета
df['lemma_accuracy'] = (df['lemma_word'].str.lower() == df['target_Lemma'].str.lower()).astype(int)

# совпадение частей речи предсказаний и таргета
df['pos_accuracy'] = (df['upos_word'] == df['target_mapped_POS']).astype(int)

In [79]:
df.head(100)

,sentence_id,sentence_text,word,probability,token_probabilities,token_count,depth,tokens,token_texts,stop_reason,...,feats,stanza_results,target_word_id,target_Lemma,target_mapped_POS,target_mapped_feats_ud,feature_accuracy,cloze_accuracy,lemma_accuracy,pos_accuracy
0,1,В тот момент <mask>,ы,0.083860,[np.float16(0.08386)],1,1,[4655],['ы'],{' '},...,None,"{'upos_word': 'PUNCT', 'lemma_word': 'ы', 'fea...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
1,1,В тот момент <mask>,в,0.023470,[np.float16(0.02347)],1,1,[5591],['в'],{' '},...,None,"{'upos_word': 'ADP', 'lemma_word': 'в', 'feats...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
2,1,В тот момент <mask>,в,0.016000,[np.float16(0.016)],1,1,[111072],['\xa0в'],{' '},...,None,"{'upos_word': 'ADP', 'lemma_word': 'в', 'feats...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
3,1,В тот момент <mask>,а,0.013470,[np.float16(0.01347)],1,1,[1506],['а'],{' '},...,None,"{'upos_word': 'CCONJ', 'lemma_word': 'а', 'fea...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
4,1,В тот момент <mask>,В,0.013270,[np.float16(0.01327)],1,1,[16604],['В'],{' '},...,None,"{'upos_word': 'INTJ', 'lemma_word': 'в', 'feat...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,В тот момент <mask>,все,0.001384,"[np.float16(0.016), np.float16(0.0864)]",2,2,"[111072, 102121]","['\xa0в', 'се']",{' '},...,Animacy=Inan|Case=Nom|Gender=Neut|Number=Sing|...,"{'upos_word': 'PRON', 'lemma_word': 'все', 'fe...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...","[Case=Nom, Number=Sing, Animacy=Inan]",0,0,0
96,1,В тот момент <mask>,Автор,0.001367,"[np.float16(0.001987), np.float16(0.6875)]",2,2,"[123674, 37773]","['Ав', 'тор']",{' '},...,Animacy=Anim|Case=Nom|Gender=Masc|Number=Sing,"{'upos_word': 'NOUN', 'lemma_word': 'автор', '...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...","[Case=Nom, Number=Sing]",0,0,1
97,1,В тот момент <mask>,так,0.001349,[np.float16(0.001349)],1,1,[105789],['так'],{' '},...,Degree=Pos,"{'upos_word': 'ADV', 'lemma_word': 'так', 'fea...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...",[],0,0,0
98,1,В тот момент <mask>,Ст,0.001339,[np.float16(0.001339)],1,1,[74598],['Ст'],{'.'},...,Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing,"{'upos_word': 'NOUN', 'lemma_word': 'статья', ...",атмосфера,атмосфера,NOUN,"{'NOUN', 'Animacy=Inan', 'Case=Nom', 'Gender=F...","[Case=Nom, Animacy=Inan, Number=Sing]",0,0,1


In [ ]:
df.to_csv('llama_with_all.csv')